In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, auc, precision_recall_curve, confusion_matrix
from tensorflow.keras.models import load_model

In [ ]:
# Load saved models
gb_model = joblib.load('models/gradientboost_model.pkl')
rf_model = joblib.load('models/randomforest_model.pkl')
autoencoder = load_model('models/autoencoder_model.h5')

In [ ]:
# Load dataset
df_test = pd.read_parquet("data/clean/sensor_clean.pqt")
df_test = df_test[df_test["timestamp"] >= "2018-06-01"]
X_test = df_test.drop(columns=["timestamp", "machine_status_code"])
y_test = df_test["machine_status_code"].replace({2: 1})  # Convert RECOVERING to anomaly

In [ ]:
# Predict with GradientBoost
y_pred_gb = gb_model.predict(X_test)
# Predict with RandomForest
y_pred_rf = rf_model.predict(X_test)

In [ ]:
# Autoencoder Predictions
X_test_scaled = (X_test - X_test.min()) / (X_test.max() - X_test.min())  # Min-Max Scaling
X_test_pred = autoencoder.predict(X_test_scaled)
reconstruction_errors = np.mean(np.abs(X_test_scaled - X_test_pred), axis=1)
best_threshold = np.percentile(reconstruction_errors, 95)
y_pred_autoencoder = (reconstruction_errors > best_threshold).astype(int)

In [ ]:
# Compute Metrics
def compute_metrics(y_true, y_pred, model_name):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    prec_vals, rec_vals, _ = precision_recall_curve(y_true, y_pred)
    auc_pr = auc(rec_vals, prec_vals)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return pd.DataFrame({
        "Model": [model_name], "Accuracy": [acc], "Precision": [prec], "Recall": [rec], "F1 Score": [f1],
        "AUC-PR": [auc_pr], "True Negatives": [tn], "False Positives": [fp], "False Negatives": [fn], "True Positives": [tp]
    })

In [ ]:
# Evaluate All Models
metrics_gb = compute_metrics(y_test, y_pred_gb, "GradientBoost")
metrics_rf = compute_metrics(y_test, y_pred_rf, "RandomForest")
metrics_autoencoder = compute_metrics(y_test, y_pred_autoencoder, "Autoencoder")

In [ ]:
# Combine Results
metrics_df = pd.concat([metrics_gb, metrics_rf, metrics_autoencoder], ignore_index=True)
print(metrics_df)

In [ ]:
# Visualization
plt.figure(figsize=(10, 5))
sns.barplot(x="Model", y="F1 Score", data=metrics_df, palette='Blues_d')
plt.title("F1 Scores of Models")
plt.xlabel("Model")
plt.ylabel("F1 Score")
plt.show()